<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 5 &middot; Make it better</div><div style="font-size:15px;color:#cfd4ff;">The hill climb &middot; about 14 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 5. Make it better (v1 to v2)

**Time:** about 14 minutes.

**The question we're starting with:** how do I know my agent really got *better*, and not just *different*?

In Module 4 we found problems by reading. That doesn't scale, and "it feels better" isn't proof. This time we'll do two things that do:

1. **Define "good" as a number.** We'll build a **rubric evaluator** from our own traces and gap list, then score our agents with it.
2. **Let the agent optimizer climb for us.** It rewrites the instructions, tries several candidates, and scores each with *our* rubric.

The key idea: **the optimizer is only as trustworthy as the rubric guiding it.** A bad ruler makes a confident, wrong climb.

**Promotion rule, declared before the scores:** require complete coverage, a positive gain beyond the observed repeat spread on both sets, no category or guard regression, and human review. Otherwise retain v1. This is a conservative teaching policy, not statistical proof or a production release rule.

| Our name | Foundry version | What changed |
|---|---|---|
| v0 | Pinned by setup | Generic instructions |
| v1 | Pinned in Module 4 | Better instructions |
| v2 checkpoint | Pinned after measurement | Promote the candidate only if it passes; otherwise keep v1 |


---

## 5.1 Which versions do I have to compare?

We'll need the agent's saved versions, so let's list them and give them our lab names.

In [ ]:
import sys
import uuid
import hashlib
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))
import lab

project_client, openai_client = lab.connect()
dev_cases, test_cases = lab.load_cases("dev"), lab.load_cases("test")

state = lab.load_state()
if not {"v0_version", "v1_version", "our_gaps"} <= state.keys():
    raise RuntimeError("Finish notebook 04, including the gap list, before starting this notebook.")
versions = {str(v.version): v for v in project_client.agents.list_versions(lab.AGENT_NAME)}
v0, v1 = versions[state["v0_version"]], versions[state["v1_version"]]
print(f"v0 = Foundry version {v0.version}   v1 = Foundry version {v1.version}")
print(f"{len(dev_cases)} dev questions, {len(test_cases)} test questions")

## 5.2 How do I turn "good" into a number?

A **rubric evaluator** is an AI judge with a checklist. It reads a question and an answer and gives a score from 0 to 1 (higher is better).

We could write the checklist ourselves, and Foundry can also **draft one from our agent's real traces**. We'll give it two ingredients:

- the traces from Module 4 (what the agent really did), and
- the gap list we wrote (what we *think* matters).

**Before you run it:** read the gaps saved in Module 4. We reuse that list rather than asking you to paste it again. Placeholder text is rejected before it reaches the service.

⏱️ **Long wait: about 1 minute.** While it runs, open one of the [More labs](../more/README.md) in replay mode.

In [ ]:
our_gaps = state["our_gaps"]
if not our_gaps or any("replace me" in gap.lower() or not gap.strip() for gap in our_gaps):
    raise ValueError("Return to Module 4 and save the gaps we actually observed.")
rubric_name = state.get("rubric_name") or f"trailmate-rubric-{uuid.uuid4().hex[:6]}"
lab.save_state(rubric_name=rubric_name)

from azure.ai.projects.models import (
    EvaluatorGenerationInputs,
    PromptEvaluatorGenerationJobSource,
    TracesEvaluatorGenerationJobSource,
)

job = EvaluatorGenerationInputs(
    sources=[
        # Real conversations from the last day, so the rubric is grounded in what happened.
        TracesEvaluatorGenerationJobSource(
            agent_name=lab.AGENT_NAME,
            start_time=datetime.fromisoformat(state["practice_started_at"]),
            description="Traces from our TrailMate practice questions",
        ),
        # Our own words about what matters, so the judge looks for the right things.
        PromptEvaluatorGenerationJobSource(
            prompt="Problems we found with this agent: " + "; ".join(our_gaps),
            description="Our gap list from Module 4",
        ),
    ],
    model=lab.FRONTIER_MODEL,  # the model that writes the rubric
    evaluator_name=rubric_name,
    evaluator_display_name="Our TrailMate rubric",
)

if "rubric_version" in state:
    rubric = project_client.evaluators.get_version(name=rubric_name, version=state["rubric_version"])
else:
    poller = project_client.evaluators.begin_create_generation_job(job=job, operation_id=rubric_name)
    rubric = lab.wait_for_poller(poller, "Rubric generation")
    lab.save_state(rubric=rubric.name, rubric_version=str(rubric.version))
print("Rubric created:", rubric.name, "version", rubric.version)

## 5.3 Can I trust the rubric I just generated?

Never use a ruler you haven't looked at. Let's read what Foundry wrote, then read **our reference rubric** side by side.

In [ ]:
# What Foundry drafted from our traces and gaps.
print(rubric.definition)

And here is our reference rubric, written by hand, to compare against.

In [ ]:
# Our reference rubric, written by hand for comparison.
print((lab.DATA / "evaluators" / "trailmate-quality.yaml").read_text(encoding="utf-8"))

**Think:**
- What does the generated rubric check that ours doesn't, and the other way round?
- Does it catch "water-resistant" being upgraded to "waterproof"?
- Does it punish an agent for answering something it should have declined?

If the rubric misses something important, our scores will be blind to it. That's the part worth slowing down for. The reference file describes a 1-to-5 judge; the generated rubric reports a normalized 0-to-1 aggregate. They are not interchangeable scores.

**Check the evidence contract too.** A criterion about retrieval cannot be judged from the final answer alone. Below we send the structured agent output, including tool calls and retrieved results. The optimizer also evaluates the agent workflow. Inspect the per-dimension reasons on the first batch before trusting either score. Human answer keys in our JSONL files remain useful for review; merely uploading them does not make the judge read them.

## 5.4 How do I score every version the same way?

Foundry needs our questions as **datasets**. We upload `dev` (used to tune and score) and `test` (the held-out set the optimizer *never* sees). Then we create an **evaluation** that ties our rubric to a dataset format.

In [ ]:
from azure.core.exceptions import ResourceExistsError


def upload(split):
    """Upload one question set as a versioned dataset, reusing it if it's already there."""
    name = f"trailmate-{split}"
    version = hashlib.sha256((lab.DATA / f"{split}.jsonl").read_bytes()).hexdigest()[:12]
    try:
        return project_client.datasets.upload_file(name=name, version=version, file_path=str(lab.DATA / f"{split}.jsonl"))
    except ResourceExistsError:
        return project_client.datasets.get(name=name, version=version)


dev_dataset, test_dataset = upload("dev"), upload("test")
print("Datasets:", dev_dataset.name, "and", test_dataset.name)

Next, the evaluation itself. It ties our rubric to the shape of our questions, and we'll reuse it for every version so every score comes from the same ruler.

In [ ]:
from openai.types.eval_create_params import DataSourceConfigCustom

# One evaluation = our questions' shape + our rubric. We reuse it for every version,
# so every score comes from the same ruler.
evaluation_options = dict(
    name="TrailMate rubric evaluation",
    data_source_config=DataSourceConfigCustom(
        type="custom",
        item_schema={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        include_sample_schema=True,
    ),
    testing_criteria=[
        {
            "type": "azure_ai_evaluator",
            "name": "Our rubric",
            "evaluator_name": rubric.name,
            "evaluator_version": str(rubric.version),
            "initialization_parameters": {"deployment_name": lab.FRONTIER_MODEL},
            # The rubric checks retrieval, so it needs the structured output, not just final text.
            "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_items}}"},
        }
    ],
)
state = lab.load_state()
evaluation = (openai_client.evals.retrieve(state["evaluation_id"]) if "evaluation_id" in state
              else openai_client.evals.create(**evaluation_options))
print("Evaluation ready:", evaluation.id)

# Remember these so the next notebook can pick up where we left off.
lab.save_state(
    rubric=rubric.name,
    rubric_version=str(rubric.version),
    evaluation_id=evaluation.id,
    dev_dataset_id=dev_dataset.id,
    test_dataset_id=test_dataset.id,
)

## 5.5 How high are we right now?

Now we score v0 and v1 on the **dev** set. Each is run **twice**. The difference between repeats (the **spread**) is a small variability check, not a confidence interval or proof of statistical significance.

Every repeat must score all twelve questions. An errored or missing item makes the batch **invalid**, not cheaper or better. Run IDs are saved immediately; rerunning this cell resumes them. After fixing a failed run, use an explicit new `attempt` rather than repeatedly submitting work.

> **While you wait:** a judge can only score what it's shown. Our rubric checks retrieval, so the evaluation sends the judge the agent's search calls and results, not just the final answer. With only the answer, could it tell a grounded "water-resistant, not waterproof" from a lucky guess? When the batch finishes, read a few per-case reasons: does the judge point to manual evidence?

**Predict:** how much higher will v1 score than v0?

⏱️ **Long wait: about 3 minutes.** Read the **While you wait** box above, or open one of the [More labs](../more/README.md) in replay mode.

In [ ]:
baseline = [
    lab.run_repeated_eval(openai_client, evaluation.id, dev_dataset.id, lab.AGENT_NAME, v.version, label, cases=dev_cases)
    for v, label in [(v0, "v0 on dev"), (v1, "v1 on dev")]
]

scoreboard = pd.DataFrame(
    [{"version": r["label"], "score": r["score"], "spread": r["spread"], "tokens": r["tokens"], "errored": r["errored"]}
     for r in baseline]
)
display(scoreboard.round(3))
lab.require_valid(*baseline)
print("Evidence files:", [r["artifact"] for r in baseline])

**Expected:** two rows (v0 and v1), each with a score, a spread, a token count and `errored` at 0.

**Read it carefully.**
- Did v1 beat v0? By more than the spread? A gain smaller than the spread could just be noise.
- Any `errored` items? Those weren't scored, so that version was graded on fewer questions.

Now the useful view: **which of our six ways are weakest?**

In [ ]:
by_way = pd.DataFrame({r["label"]: lab.scores_by_way(r["per_item"], dev_cases) for r in baseline})
by_way

**Think:** which row is lowest? That's where the optimizer has the most room to help, and where a good rewrite should show up.

## 5.6 Can the optimizer climb for me?

The **agent optimizer** does what you'd do by hand, only faster and wider. It rewrites the instructions, tries several **candidates**, scores each on a set of questions with our rubric, and reports back.

Three choices are ours:

- **Training set:** the questions it learns from. We give it `dev`. It never sees `test`.
- **Judge:** our rubric.
- **Budget:** how many candidates to try. More tries cost more and may climb higher.

**Go deeper:** the optimizer proposes candidate rewrites, scores each one on the training questions with our rubric, and reports the best. `max_candidates` caps how many get fully evaluated, which is the main lever on cost.

First, a sensible habit: **ask what it will cost before we spend it.**

In [ ]:
from azure.ai.projects.models import (
    AgentOptimizationCandidateSearchConfiguration,
    AgentOptimizationConfiguration,
    AgentOptimizationEstimateInputs,
    AgentOptimizationEvaluationConfiguration,
    AgentOptimizationEvaluator,
    AgentOptimizationFoundryAgentTargetConfiguration,
    AgentOptimizationJob,
    AgentOptimizationModelConfiguration,
    AgentOptimizationSpace,
    AgentOptimizationTargetCompletionDatasetReferenceDataSource,
    AgentOptimizationTargetCompletionEvaluationSet,
    EvaluationModelConfiguration,
)

target = AgentOptimizationFoundryAgentTargetConfiguration(name=lab.AGENT_NAME, version=str(v1.version))
writer = AgentOptimizationModelConfiguration(model=lab.FRONTIER_MODEL)  # the model that rewrites instructions

settings = AgentOptimizationConfiguration(
    evaluation_configuration=AgentOptimizationEvaluationConfiguration(
        training_set=AgentOptimizationTargetCompletionEvaluationSet(
            source=AgentOptimizationTargetCompletionDatasetReferenceDataSource(name=dev_dataset.name, version=dev_dataset.version)
        ),
        evaluators=[AgentOptimizationEvaluator(name=rubric.name, version=str(rubric.version))],
        evaluation_model=EvaluationModelConfiguration(model=lab.FRONTIER_MODEL),
    ),
    candidate_search_configuration=AgentOptimizationCandidateSearchConfiguration(max_candidates=3),
    agent_optimization_space=AgentOptimizationSpace(target_attributes=["instructions"]),  # one lever: instructions
)

estimate = project_client.agents.estimate_optimization_job(
    AgentOptimizationEstimateInputs(
        optimization_model_configuration=writer, optimization_configuration=settings, target_configuration=target
    )
)
print(estimate)

**Expected:** an estimate with call counts and a cost range.

**Think:** is that cost sensible for what we expect to gain? Notice we limited it to **three candidates** and **one lever** (instructions). We're changing one thing at a time, even though the optimizer could do more.

Ready? Start the job once. The helper saves and resumes its ID, reports progress, and stops waiting after twenty minutes without cancelling or resubmitting it.

> **While you wait:** this estimate is the cost of *running the optimizer once*. It isn't what the improved agent will cost to serve every day. Open `data/pricing.json`: which prices would we need before anyone could claim the new version saves money?

⏱️ **Long wait: about 11 minutes.** Read the **While you wait** box above, or open one of the [More labs](../more/README.md) in replay mode.

In [ ]:
state = lab.load_state()
job_context = {"v1": str(v1.version), "rubric": rubric.name, "version": str(rubric.version),
               "settings": settings.as_dict(), "writer": writer.as_dict()}
if "optimization_job_id" in state:
    if state["optimization_context"] != job_context:
        raise RuntimeError("Saved optimizer settings differ. Finish this experiment before starting a new one.")
    job_id = state["optimization_job_id"]
else:
    poller = project_client.agents.begin_create_optimization_job(
        AgentOptimizationJob(target_configuration=target, optimization_model_configuration=writer,
                             optimization_configuration=settings, display_name="TrailMate: improve instructions")
    )
    job_id = poller.details["job_id"]
    lab.save_state(optimization_job_id=job_id, optimization_context=job_context)
print("Optimization job:", job_id)
result = lab.wait_for_optimization_job(project_client, job_id)
print("Finished. Best score:", result.candidate_summary.best_score, "| baseline:", result.candidate_summary.baseline_score)

## 5.7 What did the optimizer actually change?

Never promote blindly. A candidate is a *proposal*. Let's see each one, its score, and **what it actually changed**.

In [ ]:
candidates = list(project_client.agents.list_optimization_candidates(job_id, expand=["mutations"]))

rows = []
for c in candidates:
    new_text = next((m.value for m in (c.output.mutations if c.output else []) if m.type == "instructions"), None)
    rows.append({
        "candidate": c.name,
        "status": c.status,
        "score": c.evaluation.score if c.evaluation else None,
        "avg_tokens": c.evaluation.avg_tokens if c.evaluation else None,
        "rationale": (c.rationale or "")[:90],
        "instructions": new_text,
    })

pd.DataFrame(rows).drop(columns=["instructions"])

Scores only tell us who won. To know *why*, let's read the winner's instructions in full.

In [ ]:
# Read the winning instructions in full. Do they make sense to a human?
completed = [r for r in rows if r["status"] == "completed" and r["score"] is not None and r["instructions"]]
if not completed:
    raise RuntimeError("No completed instruction candidate is available. Keep v1 and inspect the optimizer job.")
best = max(completed, key=lambda r: (r["score"], r["candidate"]))
print("Best candidate:", best["candidate"], "| score:", best["score"])
print("\n" + best["instructions"])

**Think:**
- Do these instructions tell the agent what to do when the manual is silent? When a question is off-topic?
- Could a candidate score well by **refusing too much**? That's the trap. A cautious agent never invents anything, and it's also useless.

We'll check for that trap on purpose, in a moment.

## 5.8 How do I stage a candidate without promoting it?

We create a separate candidate agent with the same model and tools. This is staging, not promotion: the main agent remains unchanged until the candidate passes our checks.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

best_candidate = next(c for c in candidates if c.name == best["candidate"])
candidate_name = f"{lab.AGENT_NAME}-optimizer-candidate"
state = lab.load_state()
if state.get("candidate_job_id") == job_id:
    if state.get("staged_candidate_id") != best_candidate.candidate_id:
        raise RuntimeError("The staged candidate is not the selected optimizer candidate. Do not mix their results.")
    v2 = next(v for v in project_client.agents.list_versions(candidate_name)
              if str(v.version) == state["candidate_version"])
else:
    v2 = project_client.agents.create_version(candidate_name, definition=PromptAgentDefinition(
        model=v1.definition.model, instructions=best["instructions"], tools=v1.definition.tools))
    lab.save_state(candidate_job_id=job_id, candidate_version=str(v2.version), staged_candidate_id=best_candidate.candidate_id)
expected_candidate = PromptAgentDefinition(model=v1.definition.model, instructions=best["instructions"], tools=v1.definition.tools)
if v2.definition.as_dict() != expected_candidate.as_dict():
    raise RuntimeError("The staged definition differs from the selected candidate. Do not evaluate or promote it.")
print(f"Staged {candidate_name}, version {v2.version}. The main agent has not been promoted.")

## 5.9 Did it get better, or just more cautious?

Our dev set holds a few **guard cases**: questions the manuals *do* answer, where the right move is to just answer. If v2 learned to hedge or refuse everything, it will fail these.

A quick look with our own eyes, before any scoring:

In [ ]:
hedges = ["not specified", "isn't specified", "doesn't say", "does not say", "don't have", "cannot", "can't"]

for case in [c for c in dev_cases if c["guard"]]:
    answer = lab.ask_agent(openai_client, case["query"], agent_name=candidate_name, agent_version=v2.version).output_text.strip()
    hedged = any(h in answer.lower() for h in hedges)
    print(f"\n[{case['id']}] {case['query']}")
    print("v2 says:", answer[:300])
    print("Hedge-word check:", "review this wording" if hedged else "no listed hedge words; still check the facts")

**Think:** a hedge-word match is only a review hint: "cannot" might be part of a correct warning. Check the actual answer against the manual. One answerable guard case does not prove broad reliability. If the rubric needs changing, start a new measurement campaign and rerun all baselines; never change it halfway through a comparison.

## 5.10 Is the improvement real?

All the scoring so far used `dev`, which the optimizer trained on. A fair test needs questions it **never saw**. That's our held-out `test` set. We score v0, v1 and v2 side by side.

⏱️ **Long wait (we're still timing this step).** While it runs, open one of the [More labs](../more/README.md) in replay mode.

In [ ]:
final = [
    lab.run_repeated_eval(openai_client, evaluation.id, test_dataset.id, name, v.version, label, cases=test_cases)
    for name, v, label in [(lab.AGENT_NAME, v0, "v0 on test"), (lab.AGENT_NAME, v1, "v1 on test"),
                           (candidate_name, v2, "v2 candidate on test")]
]

board = pd.DataFrame(
    [{"version": r["label"], "score": r["score"], "spread": r["spread"], "tokens": r["tokens"], "errored": r["errored"]}
     for r in final]
)
display(board.round(3))
lab.require_valid(*final)


That's the scoreboard. But is the gain *real*? We compare it to the noise we measured.

In [ ]:
# Two repeats are a variability check, not statistical proof.
gain = final[2]["score"] - final[1]["score"]
noise = max(final[1]["spread"], final[2]["spread"])
print(f"Candidate minus v1: {gain:.3f}. Observed repeat spread: {noise:.3f}.")
print("Worth further review." if gain > noise else "Within observed variation; do not claim an improvement.")

print(pd.DataFrame({r["label"]: lab.scores_by_way(r["per_item"], test_cases) for r in final}))

# Promotion happens only after the dev comparison and human review below.

**Expected:** one column per version (v0, v1, v2) and one row per way.

**Read the table by row.** Which of the six ways did v2 improve most? Which barely moved? Whatever barely moved is a different kind of problem. An instruction rewrite may not fix it, and that's a clue for Module 6.

## 5.11 Why do I need questions the optimizer never saw? (the aha)

One last experiment. We'll measure the *same* agent (v2) on the questions the optimizer trained on, and compare that gain with the gain on questions it never saw.

⏱️ **Long wait (we're still timing this step).** While it runs, open one of the [More labs](../more/README.md) in replay mode.

In [ ]:
# Same rubric, same agent. Only the questions differ: ones it trained on, and ones it never saw.
v2_dev = lab.run_repeated_eval(openai_client, evaluation.id, dev_dataset.id, candidate_name, v2.version,
                               "v2 candidate on dev", cases=dev_cases)
lab.require_valid(v2_dev)
dev_gain = round(v2_dev["score"] - baseline[1]["score"], 2)

print(f"v1 to v2 gain on dev (what the optimizer trained on): {dev_gain}")
print(f"v1 to v2 gain on test (questions it never saw):       {gain:.3f}")

## 5.12 Has the candidate earned promotion?

Our conservative teaching rule requires a positive gain beyond the observed repeat spread on both sets, no category or guard regression, and human review. It is not a production release policy. Do not tune this rule after seeing the scores.

Set `REVIEW_APPROVED` only after reading the instructions, guard answers and judge reasons. If the candidate does not pass, **keeping v1 is the v2 checkpoint**. Module 6 will use that actual retained baseline, not a rejected candidate.

In [ ]:
REVIEW_APPROVED = False  # Change only after reviewing the evidence, not to force a green result.

dev_delta = v2_dev["score"] - baseline[1]["score"]
checks = lab.quality_failures(v2_dev, baseline[1], dev_cases, tolerance=0.0)
checks += lab.quality_failures(final[2], final[1], test_cases, tolerance=0.0)
earned = not checks and gain > noise and dev_delta > max(v2_dev["spread"], baseline[1]["spread"])
state = lab.load_state()
was_promoted = state.get("promoted_optimizer_job") == job_id
if was_promoted and state.get("promoted_optimizer_candidate") != best_candidate.candidate_id:
    raise RuntimeError("This job already promoted a different candidate. Keep the recorded decision separate.")
approved = REVIEW_APPROVED or was_promoted
if earned and approved:
    if state.get("promoted_optimizer_job") == job_id:
        baseline_version = state["promoted_optimizer_version"]
    else:
        promoted = project_client.agents.promote_optimization_candidate(job_id, best_candidate.candidate_id)
        baseline_version = str(promoted.promotion.promoted_agent.version)
        lab.save_state(promoted_optimizer_job=job_id, promoted_optimizer_candidate=best_candidate.candidate_id,
                       promoted_optimizer_version=baseline_version)
    confirmed = next(v for v in project_client.agents.list_versions(lab.AGENT_NAME) if str(v.version) == baseline_version)
    if confirmed.definition.as_dict() != v2.definition.as_dict():
        raise RuntimeError("The promoted version differs from the one we measured. Stop and inspect it before continuing.")
    baseline_dev, baseline_test = v2_dev, final[2]
    print(f"Promoted the measured candidate: {lab.AGENT_NAME}, version {baseline_version}.")
else:
    baseline_version = str(v1.version)
    baseline_dev, baseline_test = baseline[1], final[1]
    print("No promotion. We keep v1 at the v2 checkpoint.")
    print("Reasons:", checks or ["Gain is inconclusive or human review has not been approved."])

lab.save_state(v2_version=baseline_version, v2_promoted=bool(earned and approved),
               quality_baseline_dev=baseline_dev, quality_baseline_test=baseline_test,
               candidate_test=final[2], test_scores={r["label"]: r for r in final[:2]})

> **Aha:** The gain on questions the optimizer trained on is the gain it *wanted*. The gain on questions it never saw is the gain we can *trust*. If the second number is smaller, that's normal, and it's exactly why a held-out set isn't optional. "Better" only counts when it survives data the climber couldn't see.

**Picture it:** the exam questions you didn't study from. Acing the practice paper proves you studied the practice paper; the unseen exam shows what you actually learned.

> 💼 **What would I tell my manager?** The new version only counts as better if it improves on questions it never saw while being tuned. If that gain is small or missing, we keep the version we have.

We also took away:

- **A rubric turns opinion into a number.** Without it, the optimizer has no direction.
- **The optimizer is a proposer, not a judge.** We read its candidates and checked for over-correction before trusting it.
- **Held-out data keeps us honest.** We tuned on `dev` and checked one selected candidate on `test`; twelve cases are evidence, not a production guarantee.
- **Noise is real.** A gain smaller than the spread isn't a win yet.
- A good climb is **measured, inspected and then promoted**, in that order.

**Break, fix, learn:** an optimizer can win by being timid. Guard cases catch that.

**Go further (optional):** raise `max_candidates` to 5 and run the optimizer again. Check the estimate first. Does the bigger search find something better, or just spend more?

**Next:** `06-make-it-optimal.ipynb`. Whether we promoted the rewrite or kept v1, we have a measured baseline. Can a router improve its cost or latency without unacceptable quality loss?